# Lawyer AI: Comparing Multiple RAG Types (Local Ollama)

This notebook builds and compares multiple Retrieval-Augmented Generation (RAG) strategies over the same `law_data/` corpus used in `law.ipynb`.

Implemented RAG variants:
- Basic Dense RAG
- Hybrid RAG (Dense + BM25)
- Multi-Query RAG
- Corrective RAG (fallback when retrieval confidence is low)

All methods answer as a legal assistant with source citations.

In [1]:
# Install once if needed
# %pip install -q openai pypdf numpy

In [2]:
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Dict
from collections import Counter, defaultdict
import math
import re
import time

import numpy as np
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import display, Markdown

In [3]:
# ---- Config ----
DATA_DIR = Path("../law_data") if Path("../law_data").exists() else Path("law_data")
LLM_MODEL = "llama3.2"
EMBED_MODEL = "nomic-embed-text"
CHUNK_SIZE = 900
CHUNK_OVERLAP = 150
TOP_K = 4
MAX_CANDIDATES_FOR_RERANK = 8

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

print("Data dir:", DATA_DIR.resolve())
print("Using chat model:", LLM_MODEL)
print("Using embedding model:", EMBED_MODEL)

Data dir: C:\Users\Arjun V P\projects\llm_engineering\law_data
Using chat model: llama3.2
Using embedding model: nomic-embed-text


In [4]:
@dataclass
class Chunk:
    source: str
    index: int
    text: str


def read_pdf_text(pdf_path: Path) -> str:
    reader = PdfReader(str(pdf_path))
    pages = []
    for page in reader.pages:
        pages.append(page.extract_text() or "")
    return "\n".join(pages)


def split_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    text = " ".join(text.split())
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = max(0, end - overlap)
    return chunks


def load_chunks(data_dir: Path) -> List[Chunk]:
    all_chunks: List[Chunk] = []

    for pdf in sorted(data_dir.glob("*.pdf")):
        raw = read_pdf_text(pdf)
        for i, ch in enumerate(split_text(raw)):
            all_chunks.append(Chunk(source=pdf.name, index=i, text=ch))

    for txt in sorted(data_dir.glob("*.txt")):
        raw = txt.read_text(encoding="utf-8", errors="ignore")
        for i, ch in enumerate(split_text(raw)):
            all_chunks.append(Chunk(source=txt.name, index=i, text=ch))

    return all_chunks


def normalize_rows(arr: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(arr, axis=1, keepdims=True) + 1e-12
    return arr / norms


def embed_with_ollama(texts: List[str], batch_size: int = 32) -> np.ndarray:
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        res = client.embeddings.create(model=EMBED_MODEL, input=batch)
        vectors.extend([d.embedding for d in res.data])
    return normalize_rows(np.array(vectors, dtype=np.float32))


def embed_with_sentence_transformers(texts: List[str]) -> np.ndarray:
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer("all-MiniLM-L6-v2")
    vecs = model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
    return np.array(vecs, dtype=np.float32)


EMBEDDING_BACKEND = "ollama"


def embed_texts(texts: List[str], batch_size: int = 32) -> np.ndarray:
    global EMBEDDING_BACKEND
    try:
        EMBEDDING_BACKEND = "ollama"
        return embed_with_ollama(texts, batch_size=batch_size)
    except Exception as exc:
        EMBEDDING_BACKEND = "sentence-transformers"
        print(
            "Falling back to local sentence-transformers embeddings because Ollama embeddings failed:",
            exc,
        )
        return embed_with_sentence_transformers(texts)


def embed_query(query: str) -> np.ndarray:
    if EMBEDDING_BACKEND == "ollama":
        q = client.embeddings.create(model=EMBED_MODEL, input=[query]).data[0].embedding
        qv = np.array(q, dtype=np.float32)
        return qv / (np.linalg.norm(qv) + 1e-12)

    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer("all-MiniLM-L6-v2")
    qv = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    return np.array(qv, dtype=np.float32)


def tokenize(text: str) -> List[str]:
    return re.findall(r"[a-zA-Z0-9]{2,}", text.lower())

In [5]:
# Build shared corpus and indexes once for all RAG variants
chunks = load_chunks(DATA_DIR)
if not chunks:
    raise ValueError(f"No .pdf or .txt files found in {DATA_DIR.resolve()}")

chunk_texts = [c.text for c in chunks]
chunk_embeddings = embed_texts(chunk_texts)

N = len(chunks)
doc_lens = []
tf_per_doc = []
df = defaultdict(int)

for txt in chunk_texts:
    toks = tokenize(txt)
    tf = Counter(toks)
    tf_per_doc.append(tf)
    doc_lens.append(len(toks))
    for t in tf.keys():
        df[t] += 1

avg_doc_len = (sum(doc_lens) / len(doc_lens)) if doc_lens else 1.0

print(f"Loaded {len(chunks)} chunks from {len(set(c.source for c in chunks))} document(s).")
print("Embeddings shape:", chunk_embeddings.shape)
print("Embedding backend:", EMBEDDING_BACKEND)
print("BM25 docs:", N, "avg tokens/chunk:", round(avg_doc_len, 2))

Loaded 1683 chunks from 3 document(s).
Embeddings shape: (1683, 768)
Embedding backend: ollama
BM25 docs: 1683 avg tokens/chunk: 138.59


In [6]:
def bm25_scores(query: str, k1: float = 1.5, b: float = 0.75) -> np.ndarray:
    q_terms = tokenize(query)
    if not q_terms:
        return np.zeros(N, dtype=np.float32)

    scores = np.zeros(N, dtype=np.float32)
    q_counts = Counter(q_terms)

    for i in range(N):
        tf = tf_per_doc[i]
        dl = doc_lens[i] if doc_lens[i] else 1
        denom_norm = k1 * (1 - b + b * dl / avg_doc_len)

        s = 0.0
        for term, qf in q_counts.items():
            f = tf.get(term, 0)
            if f == 0:
                continue
            n_qi = df.get(term, 0)
            idf = math.log(1 + (N - n_qi + 0.5) / (n_qi + 0.5))
            numer = f * (k1 + 1)
            s += idf * (numer / (f + denom_norm)) * qf

        scores[i] = s

    return scores


def topk_from_scores(scores: np.ndarray, top_k: int) -> List[Tuple[float, Chunk]]:
    idx = np.argsort(scores)[-top_k:][::-1]
    return [(float(scores[i]), chunks[i]) for i in idx]


def dense_retrieve(query: str, top_k: int = TOP_K) -> List[Tuple[float, Chunk]]:
    qv = embed_query(query)
    scores = chunk_embeddings @ qv
    return topk_from_scores(scores, top_k)


def hybrid_retrieve(query: str, top_k: int = TOP_K, alpha: float = 0.65) -> List[Tuple[float, Chunk]]:
    qv = embed_query(query)
    dense = chunk_embeddings @ qv
    sparse = bm25_scores(query)

    dense_min, dense_max = float(np.min(dense)), float(np.max(dense))
    sparse_min, sparse_max = float(np.min(sparse)), float(np.max(sparse))

    dense_n = (dense - dense_min) / (dense_max - dense_min + 1e-12)
    sparse_n = (sparse - sparse_min) / (sparse_max - sparse_min + 1e-12)

    scores = alpha * dense_n + (1 - alpha) * sparse_n
    return topk_from_scores(scores, top_k)


def generate_query_variants(query: str) -> List[str]:
    prompt = (
        "Create 3 alternative legal search queries for the same intent. Return one query per line only.\n"
        f"Original query: {query}"
    )
    try:
        rsp = client.chat.completions.create(
            model=LLM_MODEL,
            temperature=0.2,
            messages=[
                {"role": "system", "content": "You rewrite legal search queries."},
                {"role": "user", "content": prompt},
            ],
        )
        lines = [x.strip(" -\t") for x in rsp.choices[0].message.content.splitlines()]
        lines = [x for x in lines if x]
        return [query] + lines[:3]
    except Exception:
        # Fallback if local LLM query rewrite is unavailable
        return [query, f"legal meaning of {query}", f"constitutional interpretation of {query}"]


def multi_query_retrieve(query: str, top_k: int = TOP_K) -> List[Tuple[float, Chunk]]:
    variants = generate_query_variants(query)
    aggregate: Dict[Tuple[str, int], float] = {}

    for q in variants:
        for score, ch in dense_retrieve(q, top_k=top_k):
            key = (ch.source, ch.index)
            aggregate[key] = max(aggregate.get(key, -1e9), score)

    ranked = sorted(aggregate.items(), key=lambda kv: kv[1], reverse=True)[:top_k]
    out = []
    index_lookup = {(c.source, c.index): c for c in chunks}
    for key, score in ranked:
        out.append((float(score), index_lookup[key]))
    return out


def corrective_retrieve(query: str, top_k: int = TOP_K, confidence_threshold: float = 0.30) -> List[Tuple[float, Chunk]]:
    primary = dense_retrieve(query, top_k=top_k)
    best = primary[0][0] if primary else -1.0
    if best >= confidence_threshold:
        return primary

    # Corrective fallback: broaden and combine sparse + dense
    return hybrid_retrieve(query, top_k=max(top_k, 6), alpha=0.55)[:top_k]

In [7]:
def build_context(retrieved: List[Tuple[float, Chunk]]) -> str:
    blocks = []
    for score, ch in retrieved:
        blocks.append(f"[source={ch.source} chunk={ch.index} score={score:.4f}]\n{ch.text}")
    return "\n\n".join(blocks)


def answer_from_retrieved(query: str, retrieved: List[Tuple[float, Chunk]], style: str = "lawyer") -> str:
    context = build_context(retrieved)

    if style == "lawyer":
        system_prompt = (
            "You are a careful legal assistant. Use only the provided context when possible. "
            "If context is insufficient, clearly state what is missing and avoid fabrication."
        )
    else:
        system_prompt = "You are a helpful assistant."

    user_prompt = (
        f"Question:\n{query}\n\n"
        f"Retrieved context:\n{context}\n\n"
        "Answer with concise bullet points and cite source chunk tags like [source=... chunk=...]."
    )

    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=0.1,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response.choices[0].message.content


def answer_basic_rag(query: str, top_k: int = TOP_K):
    retrieved = dense_retrieve(query, top_k=top_k)
    return answer_from_retrieved(query, retrieved), retrieved


def answer_hybrid_rag(query: str, top_k: int = TOP_K):
    retrieved = hybrid_retrieve(query, top_k=top_k)
    return answer_from_retrieved(query, retrieved), retrieved


def answer_multi_query_rag(query: str, top_k: int = TOP_K):
    retrieved = multi_query_retrieve(query, top_k=top_k)
    return answer_from_retrieved(query, retrieved), retrieved


def answer_corrective_rag(query: str, top_k: int = TOP_K):
    retrieved = corrective_retrieve(query, top_k=top_k)
    return answer_from_retrieved(query, retrieved), retrieved

In [8]:
RAG_METHODS = {
    "basic_dense": answer_basic_rag,
    "hybrid_dense_bm25": answer_hybrid_rag,
    "multi_query": answer_multi_query_rag,
    "corrective": answer_corrective_rag,
}


def compare_rag_methods(query: str, top_k: int = TOP_K):
    results = {}

    for name, fn in RAG_METHODS.items():
        t0 = time.perf_counter()
        answer, retrieved = fn(query, top_k=top_k)
        dt = time.perf_counter() - t0

        results[name] = {
            "latency_sec": round(dt, 3),
            "top_sources": [f"{ch.source}#{ch.index}" for _, ch in retrieved[:3]],
            "answer": answer,
            "retrieved": retrieved,
        }

    return results


def render_comparison(query: str, top_k: int = TOP_K):
    results = compare_rag_methods(query, top_k=top_k)

    header = [
        f"## Query\n{query}\n"
    ]
    display(Markdown("\n".join(header)))

    table_lines = [
        "| RAG Type | Latency (s) | Top Retrieved Sources |",
        "|---|---:|---|"
    ]
    for name, info in results.items():
        srcs = ", ".join(info["top_sources"])
        table_lines.append(f"| {name} | {info['latency_sec']} | {srcs} |")

    display(Markdown("\n".join(table_lines)))

    for name, info in results.items():
        display(Markdown(f"### {name} answer\n{info['answer']}"))

    return results

In [9]:
# Single-query comparison
query = "What are the Fundamental Rights described in this constitution text?"
comparison = render_comparison(query, top_k=4)

## Query
What are the Fundamental Rights described in this constitution text?


| RAG Type | Latency (s) | Top Retrieved Sources |
|---|---:|---|
| basic_dense | 5.985 | constitution_english.pdf#411, constitution_english.pdf#568, constitution_english.pdf#279 |
| hybrid_dense_bm25 | 2.507 | ChildRightsandProtection-English (Final).pdf.pdf#108, constitution_english.pdf#75, constitution_english.pdf#824 |
| multi_query | 6.959 | constitution_english.pdf#188, constitution_english.pdf#411, constitution_english.pdf#568 |
| corrective | 2.636 | constitution_english.pdf#411, constitution_english.pdf#568, constitution_english.pdf#279 |

### basic_dense answer
Unfortunately, the provided text does not explicitly mention Fundamental Rights in its entirety. However, some relevant articles can be identified:

* Article 300A: "Persons not to be deprived of property save by authority of law." [source=constitution_english.pdf chunk=568]
* No other specific Fundamental Rights are mentioned in this part of the constitution.

If you'd like to know more about Fundamental Rights in general, I can provide information on the typical rights protected under Indian Constitution.

### hybrid_dense_bm25 answer
Based on the provided context, here are the Fundamental Rights described in the Constitution of India:

• The right to reside and settle in any part of the territory of India (Article 21(e))
• The right to practise any profession, or to carry on any occupation, trade or business (Article 21(f))

Note: These rights are subject to reasonable restrictions imposed by law in the interests of:

* Sovereignty and integrity of India
* Security of the State
* Friendly relations with foreign States
* Public order
* Decency or morality
* Contempt of court
* Defamation

Source:
[source=constitution_english.pdf chunk=75 score=0.7757]

### multi_query answer
Unfortunately, the provided text does not explicitly mention Fundamental Rights in its entirety. However, some provisions can be inferred to relate to fundamental rights:

• Freedom of speech in Parliament (Article 19(2) of the Constitution)
• Protection against deprivation of property without authority of law (Article 300A)
• Freedom of trade, commerce, and intercourse throughout the territory of India (Article 301)

These provisions might be considered as fundamental rights, but it is essential to note that they are not explicitly stated in the provided text. The complete list of Fundamental Rights would require a more comprehensive review of the Constitution.

Missing context: The specific articles or sections that define the Fundamental Rights are not provided in the given text.

### corrective answer
Unfortunately, the provided text does not explicitly mention Fundamental Rights in its entirety. However, I can identify some relevant articles that may be related to Fundamental Rights:

* Article 300A: "Persons not to be deprived of property save by authority of law." [source=constitution_english.pdf chunk=568]
	+ This article seems to relate to the protection of individual rights, specifically in relation to property.
* No other specific articles on Fundamental Rights are mentioned in the provided text.

To provide a more comprehensive answer, I would need access to the full text of the constitution or additional context. If you could provide more information or clarify which parts of the constitution you would like me to analyze, I'd be happy to help further.

In [10]:
# Optional: batch evaluation on multiple legal questions
test_queries = [
    "Explain the right to equality.",
    "What protections are mentioned for children?",
    "Summarize constitutional remedies available to citizens.",
]

for q in test_queries:
    _ = render_comparison(q, top_k=4)

KeyboardInterrupt: 